In [2]:
# ============================================================
# FIX FAISS / OPENMP CRASH
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(
    "OpenMP Fix Applied"
)

OpenMP Fix Applied


In [3]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama

print(
    "Libraries Loaded Successfully"
)

Libraries Loaded Successfully


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_3378/2542142073.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

print(
    "Environment Variables Loaded"
)

Environment Variables Loaded


In [5]:
# ============================================================
# LOAD EVALUATION DATASET
# ============================================================

eval_df = pd.read_csv(
    "../notebooks/evaluation_dataset.csv"
)

print(
    "Rows:",
    len(eval_df)
)

eval_df.head()

Rows: 1000


,query,article,reference_summary
0,solar observatory aims to provide better under...,cnn nasa has postponed for one day the schedul...,solar observatory aims to provide better under...
1,mary j bliges new album my life ii,cnn seventeen years after the release of her b...,mary j bliges new album my life ii is a follow...
2,germany coach joachim low claims spain are fav...,cnn germany coach joachim low has tried to lif...,germany coach joachim low claims spain are fav...
3,presidential historians weigh in on how histor...,cnn with record low approval ratings and inten...,presidential historians weigh in on how histor...
4,chinas gold medal gymnasts cleared of competin...,cnn chinas olympic gold medal gymnasts have be...,chinas gold medal gymnasts cleared of competin...


In [6]:
# ============================================================
# VERIFY DATASET
# ============================================================

print(
    eval_df.columns.tolist()
)

['query', 'article', 'reference_summary']


In [7]:
# ============================================================
# LOAD OPENAI EMBEDDINGS
# ============================================================

embeddings = OpenAIEmbeddings(

    model="text-embedding-3-small"

)

print(
    "Embeddings Loaded Successfully"
)

Embeddings Loaded Successfully


In [8]:
# ============================================================
# LOAD FAISS VECTOR STORE
# ============================================================

vector_store = FAISS.load_local(

    "../notebooks/faiss_20k",

    embeddings,

    allow_dangerous_deserialization=True

)

print(
    "FAISS Loaded Successfully"
)

FAISS Loaded Successfully


In [9]:
# ============================================================
# TEST RETRIEVAL
# ============================================================

docs = vector_store.similarity_search(

    "artificial intelligence",

    k=5

)

print(
    "Retrieved Documents:",
    len(docs)
)

Retrieved Documents: 5


In [10]:
# ============================================================
# LOAD MISTRAL
# ============================================================

llm = ChatOllama(

    model="mistral",

    temperature=0

)

print(
    "Mistral Loaded Successfully"
)

Mistral Loaded Successfully


In [11]:
# ============================================================
# TEST MISTRAL
# ============================================================

response = llm.invoke(
    "Say Hello"
)

print(
    response.content
)

 Hello there! How can I assist you today?


In [12]:
# ============================================================
# RESET RESULTS
# ============================================================

results = []

print(
    "Results Reset Successfully"
)

Results Reset Successfully


In [13]:
# ============================================================
# DENSE MISTRAL EVALUATION PIPELINE
# ============================================================

for idx, row in eval_df.iterrows():

    if idx >= 25:
        break

    print(
        f"Processing Row {idx+1}/25"
    )

    query = str(
        row["query"]
    )

    reference_summary = str(
        row["reference_summary"]
    )

    # --------------------------------------------------------
    # DENSE RETRIEVAL
    # --------------------------------------------------------

    docs = vector_store.similarity_search(

        query,

        k=5

    )

    context = "\n\n".join(

        [

            doc.page_content

            for doc in docs

        ]

    )

    # --------------------------------------------------------
    # PROMPT
    # --------------------------------------------------------

    prompt = f"""

    You are a helpful AI assistant.

    Use ONLY the provided context.

    Context:

    {context}

    Question:

    {query}

    Answer:

    """

    # --------------------------------------------------------
    # GENERATE RESPONSE
    # --------------------------------------------------------

    response = llm.invoke(
        prompt
    )

    answer = response.content

    # --------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------

    results.append({

        "pipeline":
        "Dense Mistral",

        "model":
        "Mistral",

        "retrieval_method":
        "Dense",

        "query":
        query,

        "reference_summary":
        reference_summary,

        "generated_answer":
        answer

    })

print(
    "\nDense Mistral Evaluation Completed"
)

Processing Row 1/25
Processing Row 2/25
Processing Row 3/25
Processing Row 4/25
Processing Row 5/25
Processing Row 6/25
Processing Row 7/25
Processing Row 8/25
Processing Row 9/25
Processing Row 10/25
Processing Row 11/25
Processing Row 12/25
Processing Row 13/25
Processing Row 14/25
Processing Row 15/25
Processing Row 16/25
Processing Row 17/25
Processing Row 18/25
Processing Row 19/25
Processing Row 20/25
Processing Row 21/25
Processing Row 22/25
Processing Row 23/25
Processing Row 24/25
Processing Row 25/25

Dense Mistral Evaluation Completed


In [14]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print(
    "Total Records:",
    len(results_df)
)

results_df.head()

Total Records: 25


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Dense Mistral,Mistral,Dense,solar observatory aims to provide better under...,solar observatory aims to provide better under...,Solar observatory aims to provide a better un...
1,Dense Mistral,Mistral,Dense,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J Blige's new album ""My Life II"" is a se..."
2,Dense Mistral,Mistral,Dense,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,"Yes, according to Germany coach Joachim Low, ..."
3,Dense Mistral,Mistral,Dense,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,Presidential historians suggest that history ...
4,Dense Mistral,Mistral,Dense,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,"underage, according to the International Gymn..."


In [15]:
# ============================================================
# DATAFRAME INFO
# ============================================================

results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   pipeline           25 non-null     str  
 1   model              25 non-null     str  
 2   retrieval_method   25 non-null     str  
 3   query              25 non-null     str  
 4   reference_summary  25 non-null     str  
 5   generated_answer   25 non-null     str  
dtypes: str(6)
memory usage: 16.3 KB


In [16]:
# ============================================================
# CHECK NULL VALUES
# ============================================================

results_df.isnull().sum()

pipeline             0
model                0
retrieval_method     0
query                0
reference_summary    0
generated_answer     0
dtype: int64

In [17]:
# ============================================================
# MANUAL VALIDATION
# ============================================================

results_df[

    [

        "query",

        "reference_summary",

        "generated_answer"

    ]

]

,query,reference_summary,generated_answer
0,solar observatory aims to provide better under...,solar observatory aims to provide better under...,Solar observatory aims to provide a better un...
1,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J Blige's new album ""My Life II"" is a se..."
2,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,"Yes, according to Germany coach Joachim Low, ..."
3,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,Presidential historians suggest that history ...
4,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,"underage, according to the International Gymn..."
5,critics and viewers see stewart as victor after,critics and viewers see stewart as victor afte...,critics and viewers see Jon Stewart as the vi...
6,bahr idriss abu garda faces charges in deaths,bahr idriss abu garda faces charges in deaths ...,"Yes, Bahr Idriss Abu Garda faces charges rela..."
7,south africa beat cohosts india in world cup,south africa beat cohosts india in world cup c...,"Yes, South Africa did beat co-hosts India in ..."
8,rudy ruiz its become unfashionable to have an,rudy ruiz its become unfashionable to have an ...,"According to the provided context, Rudy Ruiz ..."
9,fabio cannavaro is to join the italian national,fabio cannavaro is to join the italian nationa...,"Yes, Fabio Cannavaro is to join the Italian n..."


In [18]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(

    "dense_mistral_eval.csv",

    index=False

)

print(
    "Dense Mistral Evaluation Results Saved Successfully"
)

Dense Mistral Evaluation Results Saved Successfully


In [19]:
# ============================================================
# VERIFY SAVED FILE
# ============================================================

saved_df = pd.read_csv(
    "dense_mistral_eval.csv"
)

print(
    saved_df.shape
)

saved_df.head()

(25, 6)


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Dense Mistral,Mistral,Dense,solar observatory aims to provide better under...,solar observatory aims to provide better under...,Solar observatory aims to provide a better un...
1,Dense Mistral,Mistral,Dense,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J Blige's new album ""My Life II"" is a se..."
2,Dense Mistral,Mistral,Dense,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,"Yes, according to Germany coach Joachim Low, ..."
3,Dense Mistral,Mistral,Dense,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,Presidential historians suggest that history ...
4,Dense Mistral,Mistral,Dense,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,"underage, according to the International Gymn..."


In [20]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "Pipeline : Dense Mistral"
)

print(
    "Rows Evaluated :",
    len(saved_df)
)

print(
    "Output File : dense_mistral_eval.csv"
)

print(
    "Evaluation Completed Successfully"
)

Pipeline : Dense Mistral
Rows Evaluated : 25
Output File : dense_mistral_eval.csv
Evaluation Completed Successfully
